In [1]:
# import os
# import re
# import warnings
# import numpy as np
# import pandas as pd
# import shap
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import log_loss, accuracy_score
# from sklearn.calibration import CalibratedClassifierCV
# import xgboost as xgb
# import lightgbm as lgb
# from catboost import CatBoostClassifier

# warnings.filterwarnings("ignore")

# # ============================================================
# # CONFIGURATION
# # ============================================================
# DATA_PATH    = '/kaggle/input/competitions/march-machine-learning-mania-2025'
# if not os.path.exists(DATA_PATH): # Fallback for local testing
#     DATA_PATH = './'
    
# OUTPUT_FILE  = 'submission_tanishq_enhanced_2025.csv'
# ELO_K        = 32
# ELO_HOME_ADV = 100
# ELO_INIT     = 1500
# ELO_CARRYOVER= 0.75
# MASSEY_SYSTEMS = ["POM","SAG","MOR","DOL","COL","AP","USA","WLK","RTH","WOL"]
# CLIP_LOW, CLIP_HIGH = 0.05, 0.95
# SEEDS        = [42, 123, 2026]

# # CLEANED FEATURES + TIE-BREAKERS 
# # (Removed: ppg, ppg_allowed, best_massey_rank, ast_to_ratio)
# DIFF_FEATURES = [
#     "elo", "win_pct", "point_diff",
#     "fg_pct", "fg3_pct", "ft_pct",
#     "off_reb_per_game", "def_reb_per_game",
#     "ast_per_game", "to_per_game", "stl_per_game", "blk_per_game",
#     "opp_fg_pct", "fga_per_game", 
#     "seed_num", "avg_massey_rank", "massey_rank_std",
#     "conf_avg_elo",
#     "fg3_rate", "margin_std", "late_win_pct"  # NEW TIE-BREAKERS
# ]

# # ============================================================
# # 1. LOAD DATA
# # ============================================================
# print("---> 1. Loading Data...")

# def load_csv(filename):
#     try:
#         return pd.read_csv(os.path.join(DATA_PATH, filename))
#     except FileNotFoundError:
#         print(f"   WARNING: {filename} not found, returning empty DataFrame")
#         return pd.DataFrame()

# data = {
#     "m_reg_compact":    load_csv("MRegularSeasonCompactResults.csv"),
#     "w_reg_compact":    load_csv("WRegularSeasonCompactResults.csv"),
#     "m_reg_detail":     load_csv("MRegularSeasonDetailedResults.csv"),
#     "w_reg_detail":     load_csv("WRegularSeasonDetailedResults.csv"),
#     "m_tourney_compact":load_csv("MNCAATourneyCompactResults.csv"),
#     "w_tourney_compact":load_csv("WNCAATourneyCompactResults.csv"),
#     "m_seeds":          load_csv("MNCAATourneySeeds.csv"),
#     "w_seeds":          load_csv("WNCAATourneySeeds.csv"),
#     "m_conferences":    load_csv("MTeamConferences.csv"),
#     "w_conferences":    load_csv("WTeamConferences.csv"),
#     "massey":           load_csv("MMasseyOrdinals.csv"),
#     "stage2":           load_csv("SampleSubmissionStage2.csv"),
# }

# # ============================================================
# # 2. ELO RATINGS
# # ============================================================
# print("\n---> 2. Computing Elo Ratings...")

# def compute_elo_ratings(compact_df):
#     if len(compact_df) == 0: return {}
#     df = compact_df.sort_values(["Season","DayNum"]).reset_index(drop=True)
#     elo, season_elo, prev_season = {}, {}, None

#     for _, row in df.iterrows():
#         season, w_team, l_team = row["Season"], row["WTeamID"], row["LTeamID"]
#         w_loc, w_score, l_score = row.get("WLoc", "N"), row["WScore"], row["LScore"]

#         if season != prev_season:
#             if prev_season is not None:
#                 for team, rating in elo.items():
#                     season_elo[(prev_season, team)] = rating
#                 for team in elo:
#                     elo[team] = ELO_INIT + ELO_CARRYOVER * (elo[team] - ELO_INIT)
#             prev_season = season

#         if w_team not in elo: elo[w_team] = ELO_INIT
#         if l_team not in elo: elo[l_team] = ELO_INIT

#         if row["DayNum"] <= 132:
#             w_elo, l_elo = elo[w_team], elo[l_team]
#             w_elo_adj = w_elo + ELO_HOME_ADV if w_loc == "H" else w_elo
#             l_elo_adj = l_elo + ELO_HOME_ADV if w_loc == "A" else l_elo
            
#             w_expected = 1.0 / (1.0 + 10.0 ** ((l_elo_adj - w_elo_adj) / 400.0))
#             mov = abs(w_score - l_score)
#             mov_mult = np.log(max(mov, 1) + 1) * (2.2 / ((w_elo - l_elo) * 0.001 + 2.2))
#             k = ELO_K * mov_mult
            
#             elo[w_team] += k * (1 - w_expected)
#             elo[l_team] -= k * (1 - w_expected)

#     if prev_season is not None:
#         for team, rating in elo.items():
#             season_elo[(prev_season, team)] = rating
#     return season_elo

# m_elo = compute_elo_ratings(data["m_reg_compact"])
# w_elo = compute_elo_ratings(data["w_reg_compact"])

# # ============================================================
# # 3. SEASON STATS & TIE-BREAKERS
# # ============================================================
# print("\n---> 3. Computing Season Stats & Advanced Tie-Breakers...")

# def compute_game_level_features(df):
#     """Calculates variance and momentum from game-by-game logs."""
#     df_w = df[["Season", "DayNum", "WTeamID", "WScore", "LScore"]].copy()
#     df_w.columns, df_w["Win"] = ["Season", "DayNum", "TeamID", "Score", "OppScore"], 1
#     df_l = df[["Season", "DayNum", "LTeamID", "LScore", "WScore"]].copy()
#     df_l.columns, df_l["Win"] = ["Season", "DayNum", "TeamID", "Score", "OppScore"], 0
    
#     games = pd.concat([df_w, df_l], ignore_index=True)
#     games["Margin"] = games["Score"] - games["OppScore"]

#     # Blowout Variance
#     margin_df = games.groupby(["Season", "TeamID"])["Margin"].std().reset_index(name="margin_std")
#     # Late Season Momentum
#     late_games = games[games["DayNum"] > 100]
#     mom_df = late_games.groupby(["Season", "TeamID"]).agg(
#         late_games=("Win", "count"), late_wins=("Win", "sum")
#     ).reset_index()
#     mom_df["late_win_pct"] = mom_df["late_wins"] / mom_df["late_games"].replace(0, 1)
    
#     return margin_df.merge(mom_df[["Season", "TeamID", "late_win_pct"]], on=["Season", "TeamID"], how="left").fillna(0)

# def compute_season_stats(detail_df, compact_df):
#     all_stats = []
    
#     # Process Detailed Data
#     if len(detail_df) > 0:
#         df = detail_df[detail_df["DayNum"] <= 132].copy()
#         adv_feats = compute_game_level_features(df)
        
#         w_stats = df.groupby(["Season","WTeamID"]).agg(
#             w_games=("WScore","count"), w_score_sum=("WScore","sum"), w_opp_score_sum=("LScore","sum"), 
#             w_fgm=("WFGM","sum"), w_fga=("WFGA","sum"), w_fgm3=("WFGM3","sum"), w_fga3=("WFGA3","sum"),
#             w_ftm=("WFTM","sum"), w_fta=("WFTA","sum"), w_or=("WOR","sum"), w_dr=("WDR","sum"), 
#             w_ast=("WAst","sum"), w_to=("WTO","sum"), w_stl=("WStl","sum"), w_blk=("WBlk","sum"),
#             w_opp_fgm=("LFGM","sum"), w_opp_fga=("LFGA","sum"),
#         ).reset_index().rename(columns={"WTeamID":"TeamID"})

#         l_stats = df.groupby(["Season","LTeamID"]).agg(
#             l_games=("LScore","count"), l_score_sum=("LScore","sum"), l_opp_score_sum=("WScore","sum"), 
#             l_fgm=("LFGM","sum"), l_fga=("LFGA","sum"), l_fgm3=("LFGM3","sum"), l_fga3=("LFGA3","sum"),
#             l_ftm=("LFTM","sum"), l_fta=("LFTA","sum"), l_or=("LOR","sum"), l_dr=("LDR","sum"), 
#             l_ast=("LAst","sum"), l_to=("LTO","sum"), l_stl=("LStl","sum"), l_blk=("LBlk","sum"),
#             l_opp_fgm=("WFGM","sum"), l_opp_fga=("WFGA","sum"),
#         ).reset_index().rename(columns={"LTeamID":"TeamID"})

#         merged = pd.merge(w_stats, l_stats, on=["Season","TeamID"], how="outer").fillna(0)
#         merged = pd.merge(merged, adv_feats, on=["Season","TeamID"], how="left").fillna(0)
        
#         merged["games"]       = merged["w_games"] + merged["l_games"]
#         merged["wins"]        = merged["w_games"]
#         merged["win_pct"]     = merged["wins"] / merged["games"].replace(0,1)
#         merged["ppg"]         = (merged["w_score_sum"] + merged["l_score_sum"]) / merged["games"]
#         merged["ppg_allowed"] = (merged["w_opp_score_sum"] + merged["l_opp_score_sum"]) / merged["games"]
#         merged["point_diff"]  = merged["ppg"] - merged["ppg_allowed"]

#         total_fga, total_fga3 = merged["w_fga"] + merged["l_fga"], merged["w_fga3"] + merged["l_fga3"]
#         merged["fg_pct"]           = (merged["w_fgm"] + merged["l_fgm"]) / total_fga.replace(0,1)
#         merged["fg3_pct"]          = (merged["w_fgm3"] + merged["l_fgm3"]) / total_fga3.replace(0,1)
#         merged["fg3_rate"]         = total_fga3 / total_fga.replace(0,1) # NEW: 3PT Attempt Rate
#         merged["ft_pct"]           = (merged["w_ftm"] + merged["l_ftm"]) / (merged["w_fta"] + merged["l_fta"]).replace(0,1)
#         merged["off_reb_per_game"] = (merged["w_or"] + merged["l_or"]) / merged["games"]
#         merged["def_reb_per_game"] = (merged["w_dr"] + merged["l_dr"]) / merged["games"]
#         merged["ast_per_game"]     = (merged["w_ast"] + merged["l_ast"]) / merged["games"]
#         merged["to_per_game"]      = (merged["w_to"] + merged["l_to"]) / merged["games"]
#         merged["stl_per_game"]     = (merged["w_stl"] + merged["l_stl"]) / merged["games"]
#         merged["blk_per_game"]     = (merged["w_blk"] + merged["l_blk"]) / merged["games"]
#         merged["opp_fg_pct"]       = (merged["w_opp_fgm"] + merged["l_opp_fgm"]) / (merged["w_opp_fga"] + merged["l_opp_fga"]).replace(0,1)
#         merged["fga_per_game"]     = total_fga / merged["games"]

#         stat_cols = ["Season","TeamID","games","win_pct","point_diff","fg_pct","fg3_pct","ft_pct",
#                      "off_reb_per_game","def_reb_per_game","ast_per_game","to_per_game","stl_per_game",
#                      "blk_per_game","opp_fg_pct","fga_per_game","fg3_rate","margin_std","late_win_pct"]
#         all_stats.append(merged[stat_cols])

#     # Process Compact Fallback Data (For older seasons)
#     seasons_with_detail = set(all_stats[0]["Season"].unique()) if all_stats else set()
#     if len(compact_df) > 0:
#         cdf = compact_df[compact_df["DayNum"] <= 132].copy()
#         missing = cdf[~cdf["Season"].isin(seasons_with_detail)]
#         if len(missing) > 0:
#             adv_feats = compute_game_level_features(missing)
#             w_c = missing.groupby(["Season","WTeamID"]).agg(w_games=("WScore","count"), w_score_sum=("WScore","sum"), w_opp_score_sum=("LScore","sum")).reset_index().rename(columns={"WTeamID":"TeamID"})
#             l_c = missing.groupby(["Season","LTeamID"]).agg(l_games=("LScore","count"), l_score_sum=("LScore","sum"), l_opp_score_sum=("WScore","sum")).reset_index().rename(columns={"LTeamID":"TeamID"})
            
#             mc = pd.merge(w_c, l_c, on=["Season","TeamID"], how="outer").fillna(0)
#             mc = pd.merge(mc, adv_feats, on=["Season","TeamID"], how="left").fillna(0)
#             mc["games"]       = mc["w_games"] + mc["l_games"]
#             mc["win_pct"]     = mc["w_games"] / mc["games"]
#             mc["point_diff"]  = ((mc["w_score_sum"] + mc["l_score_sum"]) - (mc["w_opp_score_sum"] + mc["l_opp_score_sum"])) / mc["games"]
            
#             for col in ["fg_pct","fg3_pct","ft_pct","off_reb_per_game","def_reb_per_game","ast_per_game",
#                         "to_per_game","stl_per_game","blk_per_game","opp_fg_pct","fga_per_game","fg3_rate"]:
#                 mc[col] = np.nan
#             all_stats.append(mc[stat_cols])

#     return pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()

# m_stats = compute_season_stats(data["m_reg_detail"], data["m_reg_compact"])
# w_stats = compute_season_stats(data["w_reg_detail"], data["w_reg_compact"])

# # ============================================================
# # 4 - 6. AUXILIARY DATA (Massey, Seeds, Conferences)
# # ============================================================
# print("\n---> 4. Processing Rankings, Seeds, and Conferences...")

# def compute_massey(massey_df):
#     if len(massey_df) == 0: return pd.DataFrame(columns=["Season","TeamID","avg_massey_rank","massey_rank_std"])
#     final = massey_df[massey_df.get("RankingDayNum", massey_df.get("DayNum")) == 133].copy()
#     available = set(final["SystemName"].unique())
#     systems = [s for s in MASSEY_SYSTEMS if s in available] or list(available)[:10]
#     agg = final[final["SystemName"].isin(systems)].groupby(["Season","TeamID"]).agg(
#         avg_massey_rank=("OrdinalRank","mean"), massey_rank_std=("OrdinalRank","std")
#     ).reset_index().fillna(0)
#     return agg

# def process_seeds(seeds_df):
#     if len(seeds_df) == 0: return pd.DataFrame(columns=["Season","TeamID","seed_num"])
#     df = seeds_df.copy()
#     df["seed_num"] = df["Seed"].apply(lambda x: int(re.findall(r"\d+", x)[0]))
#     return df[["Season","TeamID","seed_num"]]

# def compute_conf_strength(conf_df, season_elo):
#     if len(conf_df) == 0: return pd.DataFrame(columns=["Season","TeamID","conf_avg_elo"])
#     df = conf_df.copy()
#     df["team_elo"] = df.apply(lambda r: season_elo.get((r["Season"], r["TeamID"]), ELO_INIT), axis=1)
#     conf_strength = df.groupby(["Season","ConfAbbrev"]).agg(conf_avg_elo=("team_elo","mean")).reset_index()
#     return pd.merge(df[["Season","TeamID","ConfAbbrev"]], conf_strength, on=["Season","ConfAbbrev"], how="left")[["Season","TeamID","conf_avg_elo"]]

# massey_feats = compute_massey(data["massey"])
# m_seeds, w_seeds = process_seeds(data["m_seeds"]), process_seeds(data["w_seeds"])
# m_conf = compute_conf_strength(data["m_conferences"], m_elo)
# w_conf = compute_conf_strength(data["w_conferences"], w_elo)

# # ============================================================
# # 7 - 9. BUILD FEATURES, TRAINING DATA, AND PRED GRID
# # ============================================================
# print("\n---> 5. Assembling Training & Prediction Data...")

# def build_team_features(stats_df, season_elo, seeds_df, massey_df, conf_df):
#     features = pd.DataFrame([(s, t, e) for (s, t), e in season_elo.items()], columns=["Season","TeamID","elo"])
#     for df in [stats_df, seeds_df, massey_df, conf_df]:
#         if df is not None and len(df) > 0:
#             features = features.merge(df, on=["Season","TeamID"], how="left")
            
#     features["seed_num"] = features["seed_num"].fillna(20)
#     features["avg_massey_rank"] = features.get("avg_massey_rank", pd.Series(dtype=float)).fillna(300)
#     features["massey_rank_std"] = features.get("massey_rank_std", pd.Series(dtype=float)).fillna(0)
#     features["conf_avg_elo"]    = features.get("conf_avg_elo", pd.Series(dtype=float)).fillna(ELO_INIT)
#     return features

# m_features = build_team_features(m_stats, m_elo, m_seeds, massey_feats, m_conf)
# w_features = build_team_features(w_stats, w_elo, w_seeds, None, w_conf)

# def build_matchup_features(r1, r2):
#     return {f"{c}_diff": (r1.get(c, np.nan) - r2.get(c, np.nan)) for c in DIFF_FEATURES}

# def build_training_data(features_df, tourney_df, reg_df):
#     rows, feat_idx = [], features_df.set_index(["Season","TeamID"]).to_dict("index")
#     def add_game(season, ta, tb, ta_won):
#         t1, t2 = min(ta, tb), max(ta, tb)
#         if (season, t1) not in feat_idx or (season, t2) not in feat_idx: return
#         feat = build_matchup_features(feat_idx[(season, t1)], feat_idx[(season, t2)])
#         feat.update({"Season": season, "Team1": t1, "Team2": t2, "target": 1.0 if (ta == t1 and ta_won) or (tb == t1 and not ta_won) else 0.0})
#         rows.append(feat)

#     for _, row in tourney_df.iterrows(): add_game(row["Season"], row["WTeamID"], row["LTeamID"], True)
#     if len(reg_df) > 0:
#         for _, row in reg_df[reg_df["DayNum"] >= 100].sample(frac=0.15, random_state=42).iterrows():
#             add_game(row["Season"], row["WTeamID"], row["LTeamID"], True)
#     return pd.DataFrame(rows)

# m_train = build_training_data(m_features, data["m_tourney_compact"], data["m_reg_compact"])
# w_train = build_training_data(w_features, data["w_tourney_compact"], data["w_reg_compact"])
# all_train = pd.concat([m_train, w_train], ignore_index=True)

# feature_cols = [c for c in all_train.columns if c.endswith("_diff")]
# medians = all_train[feature_cols].median()
# all_train[feature_cols] = all_train[feature_cols].fillna(medians)

# feat_idx_all = pd.concat([m_features, w_features], ignore_index=True).set_index(["Season","TeamID"]).to_dict("index")
# sample_sub = data["stage2"].copy() if len(data["stage2"]) > 0 else pd.DataFrame({"ID": []})
# pred_rows = []
# if not sample_sub.empty:
#     sub_parsed = sample_sub["ID"].str.split("_", expand=True).astype(int)
#     for _, (s, t1, t2) in sub_parsed.iterrows():
#         pred_rows.append(build_matchup_features(feat_idx_all.get((s, t1), {}), feat_idx_all.get((s, t2), {})))

# pred_df = pd.DataFrame(pred_rows).fillna(medians)

# # ============================================================
# # 10. CALIBRATED TRAINING & INFERENCE
# # ============================================================
# print(f"\n---> 6. Training Models ({len(feature_cols)} features)...")

# def train_and_predict_calibrated(train_df, pred_df, seed):
#     X, y = train_df[feature_cols].values, train_df["target"].values
#     X_pred = pred_df[feature_cols].values if not pred_df.empty else np.array([])
    
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
#     models = {
#         "LightGBM": lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, min_child_samples=50, random_state=seed, verbose=-1),
#         "XGBoost": xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=5, random_state=seed, tree_method="hist", verbosity=0),
#         "CatBoost": CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, random_seed=seed, verbose=0),
#     }

#     cv_scores, all_oof = {}, {}
#     for name, base_model in models.items():
#         fold_scores, oof_probs = [], np.zeros(len(X))
#         for tr_idx, val_idx in skf.split(X, y):
#             calibrated_model = CalibratedClassifierCV(base_model.__class__(**base_model.get_params()), method='isotonic', cv=3)
#             calibrated_model.fit(X[tr_idx], y[tr_idx])
#             oof_probs[val_idx] = calibrated_model.predict_proba(X[val_idx])[:, 1]
#             fold_scores.append(log_loss(y[val_idx], oof_probs[val_idx]))
        
#         cv_scores[name] = fold_scores # Store all fold scores for stability check
#         all_oof[name] = oof_probs

#     best_name = min(cv_scores, key=lambda k: np.mean(cv_scores[k]))
    
#     final_preds = []
#     if len(X_pred) > 0:
#         final_model = CalibratedClassifierCV(models[best_name], method='isotonic', cv=5)
#         final_model.fit(X, y)
#         final_preds = final_model.predict_proba(X_pred)[:, 1]

#     return final_preds, cv_scores, best_name, all_oof[best_name]

# all_seed_preds, all_cv_results, avg_oof = [], [], np.zeros(len(all_train))

# for seed in SEEDS:
#     preds, cv_scores, best_name, oof = train_and_predict_calibrated(all_train, pred_df, seed=seed)
#     if len(preds) > 0: all_seed_preds.append(preds)
#     all_cv_results.append((seed, best_name, cv_scores))
#     avg_oof += oof / len(SEEDS)

# # Attach average OOF predictions to the training dataframe for diagnostics
# all_train["oof_pred"] = avg_oof

# # ============================================================
# # 11. STABILITY CHECK
# # ============================================================
# print("\n---> 7. Pipeline Stability Check...")
# print("-" * 60)
# print(f"{'Seed':<6} | {'Best Model':<12} | {'Mean LogLoss':<15} | {'Std Dev (Folds)':<15}")
# print("-" * 60)
# for seed, best_name, cv_scores in all_cv_results:
#     mean_ll = np.mean(cv_scores[best_name])
#     std_ll  = np.std(cv_scores[best_name])
#     print(f"{seed:<6} | {best_name:<12} | {mean_ll:.5f}         | {std_ll:.5f}")
# print("-" * 60)

# # ============================================================
# # 12. DIAGNOSTICS: OOF ERROR SEGMENTATION
# # ============================================================
# print("\n---> 8. OOF Error Segmentation...")

# def plot_error_segmentation(df):
#     df = df.copy()
#     # Create Deciles for Elo Difference
#     df['elo_bin'] = pd.qcut(df['elo_diff'], q=10, labels=False)
    
#     segment_stats = df.groupby('elo_bin').apply(
#         lambda x: pd.Series({
#             'LogLoss': log_loss(x['target'], x['oof_pred']),
#             'Accuracy': accuracy_score(x['target'], (x['oof_pred'] > 0.5).astype(int)),
#             'Count': len(x),
#             'Mean_Elo_Diff': x['elo_diff'].mean()
#         })
#     ).reset_index()

#     fig, ax1 = plt.subplots(figsize=(12, 6))
#     sns.barplot(data=segment_stats, x='elo_bin', y='LogLoss', color='lightcoral', ax=ax1)
#     ax1.set_ylabel('Log Loss (Lower is Better)', color='darkred')
#     ax1.set_xlabel('Elo Difference Decile (0 = Heavy Underdog, 9 = Massive Favorite)')
#     ax1.set_title("OOF Error Segmentation by Elo Difference")
    
#     ax2 = ax1.twinx()
#     sns.lineplot(data=segment_stats, x='elo_bin', y='Accuracy', color='blue', marker='o', linewidth=2, ax=ax2)
#     ax2.set_ylabel('Accuracy', color='blue')
#     plt.grid(False)
#     plt.tight_layout()
#     plt.show()

# plot_error_segmentation(all_train)

# # ============================================================
# # 13. DIAGNOSTICS: RESIDUALS & CALIBRATION
# # ============================================================
# print("\n---> 9. Residuals & Calibration Analysis...")

# def plot_residual_diagnostics(df):
#     y_true, oof_probs = df["target"].values, df["oof_pred"].values
#     residuals = y_true - oof_probs
    
#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#     sns.set_theme(style="whitegrid")

#     # 1. Binned Residuals
#     bins = np.linspace(0, 1, 21)
#     bin_indices = np.digitize(oof_probs, bins) - 1
#     bin_means = [residuals[bin_indices == i].mean() for i in range(len(bins)-1) if sum(bin_indices == i) > 0]
#     bin_centers = [bins[i] + (bins[i+1]-bins[i])/2 for i in range(len(bins)-1) if sum(bin_indices == i) > 0]
#     axes[0].scatter(bin_centers, bin_means, color='blue')
#     axes[0].axhline(0, color='red', linestyle='--')
#     axes[0].set_title("Binned Average Residuals")

#     # 2. Calibration Curve
#     from sklearn.calibration import calibration_curve
#     prob_true, prob_pred = calibration_curve(y_true, oof_probs, n_bins=15, strategy='quantile')
#     axes[1].plot(prob_pred, prob_true, marker='o', label='Pipeline', color='darkorange')
#     axes[1].plot([0, 1], [0, 1], linestyle='--', label='Perfect', color='black')
#     axes[1].set_title("Calibration Curve")
#     axes[1].legend()

#     # 3. Residuals vs Elo Diff
#     sns.regplot(x=df["elo_diff"], y=residuals, lowess=True, scatter_kws={'alpha': 0.1}, line_kws={'color': 'purple'}, ax=axes[2])
#     axes[2].axhline(0, color='black', linestyle='--')
#     axes[2].set_title("Residuals vs. Elo Diff")

#     plt.tight_layout()
#     plt.show()

# plot_residual_diagnostics(all_train)

# # ============================================================
# # 14. DIAGNOSTICS: SHAP EXPLAINER
# # ============================================================
# print("\n---> 10. SHAP Feature Importance...")

# def run_shap(df, features):
#     print("   Training fast uncalibrated model for SHAP...")
#     X, y = df[features], df["target"]
#     model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, min_child_samples=50, random_state=42, verbose=-1).fit(X, y)
    
#     X_sample = X.sample(n=min(5000, len(X)), random_state=42)
#     explainer = shap.TreeExplainer(model)
#     shap_vals = explainer.shap_values(X_sample)
    
#     # Handle version compatibility
#     if isinstance(shap_vals, list): shap_vals = shap_vals[1]
#     elif len(np.shape(shap_vals)) == 3: shap_vals = shap_vals[:, :, 1]
        
#     plt.figure(figsize=(10, 8))
#     plt.title("SHAP Summary (Includes Tie-Breakers)", pad=20)
#     shap.summary_plot(shap_vals, X_sample, max_display=15, show=False)
#     plt.tight_layout()
#     plt.show()

# run_shap(all_train, feature_cols)

# # ============================================================
# # 15. SAVE SUBMISSION
# # ============================================================
# print("\n---> 11. Saving Submission...")
# if len(all_seed_preds) > 0:
#     sample_sub["Pred"] = np.clip(np.mean(all_seed_preds, axis=0), CLIP_LOW, CLIP_HIGH)
#     sample_sub[["ID","Pred"]].to_csv(OUTPUT_FILE, index=False)
#     print(f"   Submission saved: {OUTPUT_FILE} ({len(sample_sub)} rows)")
# else:
#     print("   No prediction data provided in stage1 file. Diagnostics complete.")

# print("\n🏆 Ultimate Pipeline Complete!")

In [2]:
# import pandas as pd
# import numpy as np
# import statsmodels.api as sm
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# import matplotlib.pyplot as plt
# import seaborn as sns

# # ============================================================
# # MULTICOLLINEARITY (VIF) ANALYSIS FOR 21 FEATURES
# # ============================================================
# def analyze_vif_updated(train_df, feature_list):
#     print("---> Calculating Variance Inflation Factor (VIF)...")
    
#     # Extract the _diff columns that match our current 21 features
#     feature_cols = [f"{f}_diff" for f in feature_list if f"{f}_diff" in train_df.columns]
#     X = train_df[feature_cols].copy()
    
#     # Ensure no NaNs or Infs exist before calculation
#     X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median())
    
#     # Add a constant (intercept) for the VIF calculation
#     X_const = sm.add_constant(X)
    
#     # Calculate VIF for each feature
#     vif_data = pd.DataFrame()
#     vif_data["Feature"] = X_const.columns
#     vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) 
#                        for i in range(X_const.shape[1])]
    
#     # Filter out the constant and sort
#     vif_data = vif_data[vif_data["Feature"] != "const"]
#     vif_data = vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True)
    
#     # Display the results in the console
#     print("\nVariance Inflation Factor (VIF) Results:")
#     print("-" * 55)
#     print(vif_data.to_string(index=False))
#     print("-" * 55)
    
#     # Visualize the VIF scores
#     plt.figure(figsize=(12, 10))
#     sns.barplot(x="VIF", y="Feature", data=vif_data, palette="viridis")
    
#     # Add threshold lines
#     plt.axvline(x=5, color='orange', linestyle='--', linewidth=2, label='VIF = 5 (Moderate)')
#     plt.axvline(x=10, color='red', linestyle='--', linewidth=2, label='VIF = 10 (High)')
    
#     plt.title("Feature Multicollinearity (Updated 21 Features)", fontsize=14)
#     plt.xlabel("VIF Score", fontsize=12)
#     plt.ylabel("Features", fontsize=12)
#     plt.legend()
#     plt.tight_layout()
#     plt.show()
    
#     return vif_data

# # Run the analysis using your 21 current features
# vif_results = analyze_vif_updated(all_train, DIFF_FEATURES)

In [3]:
# print("\n---> 11. Saving Submission...")
# if len(all_seed_preds) > 0:
#     sample_sub["Pred"] = np.clip(np.mean(all_seed_preds, axis=0), CLIP_LOW, CLIP_HIGH)
#     sample_sub[["ID","Pred"]].to_csv(OUTPUT_FILE, index=False)
#     print(f"   Submission saved: {OUTPUT_FILE} ({len(sample_sub)} rows)")
# else:
#     print("   No prediction data provided in stage1 file. Diagnostics complete.")

# print("\n🏆 Ultimate Pipeline Complete!")

## OG

In [4]:
# ============================================================
# MARCH MACHINE LEARNING MANIA 2025 – STAGE 2 PIPELINE
# ============================================================

import os
import re
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = "/kaggle/input/competitions/march-machine-learning-mania-2025/"
OUTPUT_FILE = "submission_stage2.csv"

ELO_K = 32
ELO_HOME_ADV = 100
ELO_INIT = 1500
ELO_CARRYOVER = 0.75

SEEDS = [42, 123, 2026]
CLIP_LOW, CLIP_HIGH = 0.05, 0.95

MASSEY_SYSTEMS = ["POM","SAG","MOR","DOL","COL","AP","USA","WLK","RTH","WOL"]

DIFF_FEATURES = [
    "elo", "win_pct", "ppg", "ppg_allowed", "point_diff",
    "fg_pct", "fg3_pct", "ft_pct",
    "off_reb_per_game", "def_reb_per_game",
    "ast_per_game", "to_per_game",
    "opp_fg_pct", "fga_per_game",
    "seed_num", "avg_massey_rank",
    "conf_avg_elo"
]

# ============================================================
# 1. LOAD DATA
# ============================================================

def load_csv(name):
    return pd.read_csv(os.path.join(DATA_PATH, name))

print("Loading data...")

m_reg = load_csv("MRegularSeasonCompactResults.csv")
w_reg = load_csv("WRegularSeasonCompactResults.csv")

m_detail = load_csv("MRegularSeasonDetailedResults.csv")
w_detail = load_csv("WRegularSeasonDetailedResults.csv")

m_tourney = load_csv("MNCAATourneyCompactResults.csv")
w_tourney = load_csv("WNCAATourneyCompactResults.csv")

m_seeds = load_csv("MNCAATourneySeeds.csv")
w_seeds = load_csv("WNCAATourneySeeds.csv")

m_conf = load_csv("MTeamConferences.csv")
w_conf = load_csv("WTeamConferences.csv")

massey = load_csv("MMasseyOrdinals.csv")

stage2 = load_csv("SampleSubmissionStage2.csv")

# ============================================================
# 2. ELO
# ============================================================

def compute_elo(df):

    df = df.sort_values(["Season","DayNum"])
    elo = {}
    season_elo = {}
    prev_season = None

    for _, row in df.iterrows():
        season = row.Season

        if season != prev_season:
            if prev_season is not None:
                for t, r in elo.items():
                    season_elo[(prev_season, t)] = r
                for t in elo:
                    elo[t] = ELO_INIT + ELO_CARRYOVER * (elo[t] - ELO_INIT)
            prev_season = season

        w = row.WTeamID
        l = row.LTeamID

        elo.setdefault(w, ELO_INIT)
        elo.setdefault(l, ELO_INIT)

        if row.DayNum <= 132:

            w_elo = elo[w]
            l_elo = elo[l]

            w_exp = 1 / (1 + 10 ** ((l_elo - w_elo)/400))
            mov = abs(row.WScore - row.LScore)
            k = ELO_K * np.log(mov+1)

            elo[w] += k * (1 - w_exp)
            elo[l] -= k * (1 - w_exp)

    for t, r in elo.items():
        season_elo[(prev_season, t)] = r

    return season_elo

print("Computing Elo...")
m_elo = compute_elo(m_reg)
w_elo = compute_elo(w_reg)

# ============================================================
# 3. SEASON STATS
# ============================================================

def compute_stats(detail):

    df = detail[detail.DayNum <= 132].copy()

    w = df.groupby(["Season","WTeamID"]).agg(
        games=("WScore","count"),
        score=("WScore","sum"),
        opp_score=("LScore","sum")
    ).reset_index().rename(columns={"WTeamID":"TeamID"})

    l = df.groupby(["Season","LTeamID"]).agg(
        games_l=("LScore","count"),
        score_l=("LScore","sum"),
        opp_score_l=("WScore","sum")
    ).reset_index().rename(columns={"LTeamID":"TeamID"})

    merged = pd.merge(w,l,on=["Season","TeamID"],how="outer").fillna(0)

    merged["games"] = merged.games + merged.games_l
    merged["wins"] = merged.games
    merged["win_pct"] = merged.wins / merged.games.replace(0,1)

    merged["ppg"] = (merged.score + merged.score_l) / merged.games
    merged["ppg_allowed"] = (merged.opp_score + merged.opp_score_l) / merged.games
    merged["point_diff"] = merged.ppg - merged.ppg_allowed

    return merged[["Season","TeamID","win_pct","ppg","ppg_allowed","point_diff"]]

m_stats = compute_stats(m_detail)
w_stats = compute_stats(w_detail)

# ============================================================
# 4. SEEDS
# ============================================================

def process_seeds(df):
    df["seed_num"] = df.Seed.str.extract("(\d+)").astype(int)
    return df[["Season","TeamID","seed_num"]]

m_seeds = process_seeds(m_seeds)
w_seeds = process_seeds(w_seeds)

# ============================================================
# 5. BUILD TEAM FEATURES
# ============================================================

def build_features(stats, elo, seeds):

    elo_df = pd.DataFrame([(s,t,e) for (s,t),e in elo.items()],
                          columns=["Season","TeamID","elo"])

    features = elo_df.merge(stats,on=["Season","TeamID"],how="left")
    features = features.merge(seeds,on=["Season","TeamID"],how="left")

    features.fillna(0,inplace=True)
    return features

m_feat = build_features(m_stats,m_elo,m_seeds)
w_feat = build_features(w_stats,w_elo,w_seeds)

all_feat = pd.concat([m_feat,w_feat],ignore_index=True)
feat_idx = all_feat.set_index(["Season","TeamID"]).to_dict("index")

# ============================================================
# 6. BUILD TRAIN DATA
# ============================================================

def build_match(r1,r2):

    row={}
    for col in DIFF_FEATURES:
        row[col+"_diff"] = r1.get(col,0) - r2.get(col,0)
    return row

def build_train(feat_df,tourney):

    rows=[]
    idx=feat_df.set_index(["Season","TeamID"]).to_dict("index")

    for _,r in tourney.iterrows():

        s=r.Season
        t1=min(r.WTeamID,r.LTeamID)
        t2=max(r.WTeamID,r.LTeamID)

        if (s,t1) not in idx or (s,t2) not in idx:
            continue

        f=build_match(idx[(s,t1)],idx[(s,t2)])
        f["target"]=1 if r.WTeamID==t1 else 0
        rows.append(f)

    return pd.DataFrame(rows)

train_m = build_train(m_feat,m_tourney)
train_w = build_train(w_feat,w_tourney)

train = pd.concat([train_m,train_w],ignore_index=True)

feature_cols=[c for c in train.columns if c.endswith("_diff")]
train.fillna(train[feature_cols].median(),inplace=True)

# ============================================================
# 7. BUILD STAGE 2 GRID
# ============================================================

print("Building Stage 2 grid...")

sub_parsed = stage2.ID.str.split("_",expand=True).astype(int)
sub_parsed.columns=["Season","T1","T2"]

pred_rows=[]

for _,r in sub_parsed.iterrows():

    s,t1,t2=r.Season,r.T1,r.T2

    r1=feat_idx.get((s,t1))
    r2=feat_idx.get((s,t2))

    if r1 and r2:
        pred_rows.append(build_match(r1,r2))
    else:
        pred_rows.append({c+"_diff":0 for c in DIFF_FEATURES})

pred_df=pd.DataFrame(pred_rows)
pred_df.fillna(train[feature_cols].median(),inplace=True)

# ============================================================
# 8. TRAIN + CV + MULTI SEED AVERAGE
# ============================================================

def train_seed(seed):

    X=train[feature_cols].values
    y=train.target.values
    X_pred=pred_df[feature_cols].values

    skf=StratifiedKFold(5,shuffle=True,random_state=seed)

    model=lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.03,
        random_state=seed
    )

    oof=np.zeros(len(X))
    scores=[]

    for tr,va in skf.split(X,y):
        m=lgb.LGBMClassifier(
            n_estimators=3000,
            learning_rate=0.03,
            random_state=seed
        )
        m.fit(X[tr],y[tr])
        p=m.predict_proba(X[va])[:,1]
        oof[va]=p
        scores.append(log_loss(y[va],p))

    print(f"Seed {seed} CV: {np.mean(scores):.5f}")

    model.fit(X,y)
    return model.predict_proba(X_pred)[:,1]

all_preds=[train_seed(s) for s in SEEDS]
final_preds=np.mean(all_preds,axis=0)

# ============================================================
# 9. SAVE
# ============================================================

stage2["Pred"]=np.clip(final_preds,CLIP_LOW,CLIP_HIGH)
stage2[["ID","Pred"]].to_csv(OUTPUT_FILE,index=False)

print("Submission saved:",OUTPUT_FILE)
print(stage2.Pred.describe())

<>:162: SyntaxWarning: invalid escape sequence '\d'
<>:162: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_17/3255757709.py:162: SyntaxWarning: invalid escape sequence '\d'
  df["seed_num"] = df.Seed.str.extract("(\d+)").astype(int)


Loading data...
Computing Elo...
Building Stage 2 grid...
[LightGBM] [Info] Number of positive: 1713, number of negative: 1621
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002815 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1048
[LightGBM] [Info] Number of data points in the train set: 3334, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.513797 -> initscore=0.055203
[LightGBM] [Info] Start training from score 0.055203
[LightGBM] [Info] Number of positive: 1713, number of negative: 1621
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000079 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 3334, number of used features: 5
[LightGBM] [Info] [binary:BoostFro